Week 13 · Day 3 — Masked Language Modeling (BERT-Style)
Why this matters

BERT is trained differently from GPT: instead of predicting the next word, it predicts missing words inside a sentence. This Masked Language Modeling (MLM) makes it great at understanding context for classification, Q&A, and embeddings.

Theory Essentials

MLM objective: Randomly replace ~15% of tokens with [MASK] → model predicts them.

Bidirectional context: BERT uses both left and right words to fill in the mask.

Comparison to GPT: GPT looks only left-to-right, BERT looks everywhere.

Applications: Sentence classification, semantic search, Q&A, embeddings.

Hugging Face makes it easy with BertForMaskedLM.

In [81]:
# Setup
import torch
from transformers import BertTokenizer, BertForMaskedLM, DataCollatorForLanguageModeling, TextDataset, Trainer, TrainingArguments

# Load BERT base
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Tiny toy dataset
text = """The sky is blue.
The cat sits on the mat.
AI is transforming the world.
Dogs are friendly.
"""
with open("toy_bert.txt", "w") as f:
    f.write(text)

dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="toy_bert.txt",
    block_size=16
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,          # <-- MLM on
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir="./bert_toy",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,
    num_train_epochs=10,
    logging_steps=5,
    save_steps=20,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()

# Test fill-mask
masked = "The sky is [MASK]."
inputs = tokenizer(masked, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
pred_id = logits[0, mask_token_index].argmax(axis=-1)
print("Prediction:", masked.replace("[MASK]", tokenizer.decode(pred_id)))


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
c:\AI-Mastery\venv\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transfor

Step,Training Loss
5,3.619500
10,1.846900


Prediction: The sky is dark.


1) Core (10–15 min)
Task: Train on the toy dataset and test with The cat [MASK] on the mat.

In [79]:
# Test fill-mask
masked = "The cat [MASK] on the mat."
inputs = tokenizer(masked, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
pred_id = logits[0, mask_token_index].argmax(axis=-1)
print("Prediction:", masked.replace("[MASK]", tokenizer.decode(pred_id)))

Prediction: The cat sat on the mat.


2) Practice (10–15 min)
Task: Add new lines (e.g., “Dogs are friendly.”). Retrain. Test with Dogs are [MASK].

In [80]:
# Test fill-mask
masked = "Dogs are [MASK]."
inputs = tokenizer(masked, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
pred_id = logits[0, mask_token_index].argmax(axis=-1)
print("Prediction:", masked.replace("[MASK]", tokenizer.decode(pred_id)))

Prediction: Dogs are allowed.


3) Stretch (optional, 10–15 min)
Task: Increase mlm_probability to 0.3. Observe effect on training loss and predictions.
Hint: Too many masks → harder learning.

Loss with 0.15: 1.846900
Loss with 0.3: 2.568700

Mini-Challenge (≤40 min)

Build a BERT-style fill-mask demo.

Train on ≥10 sentences.

Allow user to input a masked sentence.

Show the filled prediction.

Acceptance Criteria

Works with at least 3 examples.

Predictions use both left & right context.

Short note: why MLM is good for understanding vs generation.

In [82]:
# Setup
import torch
from transformers import BertTokenizer, BertForMaskedLM, DataCollatorForLanguageModeling, TextDataset, Trainer, TrainingArguments

# Load BERT base
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Tiny toy dataset
text = """The sun sets behind the mountains every evening.

My cat sleeps on the warm laptop when I study.

Tomorrow, we will run five kilometers in the park.

Artificial intelligence is changing the way humans work.

Please hand me the red notebook on the desk.

Coffee tastes better when shared with a friend.

She whispered a secret, but nobody heard.

Robots can learn tasks by observing people.

The river flows quietly under the old stone bridge.

I forgot my umbrella, so I got wet in the rain.
"""
with open("toy_bert2.txt", "w") as f:
    f.write(text)

dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="toy_bert2.txt",
    block_size=16
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,          # <-- MLM on
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir="./bert_toy",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,
    num_train_epochs=10,
    logging_steps=5,
    save_steps=20,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()

# Test fill-mask
masked = input("Please enter a masked sentence use [MASK].")
inputs = tokenizer(masked, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
pred_id = logits[0, mask_token_index].argmax(axis=-1)
print("Prediction:", masked.replace("[MASK]", tokenizer.decode(pred_id)))


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
c:\AI-Mastery\venv\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transfor

Step,Training Loss
5,4.008700
10,2.019800
15,2.875700
20,3.407400
25,2.300300
30,2.423000
35,2.292700
40,2.264600


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Prediction: tomorrow we will meet in the park.


Notes / Key Takeaways

BERT = encoder-only; learns by filling in [MASK].

Uses bidirectional context (both left & right).

Good for understanding tasks (classification, embeddings).

MLM = different pretraining objective than GPT.

Hugging Face makes experiments simple.

Reflection

Why does MLM encourage bidirectional understanding?

In what type of tasks would GPT outperform BERT?

Because in Masked Language Modeling, the model must predict the hidden token using both left and right context. To succeed, it has to attend to information from all directions in the sentence, not just the past.

 GPT shines in generation tasks where you need fluent continuations (story writing, chatbots, summarization, code completion). BERT, being bidirectional, is better at understanding/classification tasks, but it cannot naturally generate long coherent text the way GPT can.